In [ ]:
# Cell 1: normal inference without ICL.
# TEST_SCOPE can be "full", "subset", or "single".
# For TEST_SCOPE = "subset", SPLIT_INDEX chooses a deterministic 400-example chunk.
# Use TEST_SCOPE = "single" and EXAMPLE_ID = 0 for a quick GPU smoke test.
from pathlib import Path
import importlib.util

SCRIPT_PATH = Path("ICL-demo.py").resolve()
spec = importlib.util.spec_from_file_location("icl_demo", SCRIPT_PATH)
icl = importlib.util.module_from_spec(spec)
spec.loader.exec_module(icl)

MODEL = "Qwen/Qwen2.5-7B-Instruct"
MATH_CONFIG = "algebra"
TEST_SCOPE = "full"
SPLIT_INDEX = 0
EXAMPLE_ID = 0
NUM_TEST = 500
MAX_NEW_TOKENS = 768
PRINT_EXAMPLES = 5
SAVE_RESULTS = True

no_icl_args = icl.make_experiment_args(
    mode="no_icl",
    model=MODEL,
    math_config=MATH_CONFIG,
    test_scope=TEST_SCOPE,
    split_index=SPLIT_INDEX,
    example_id=EXAMPLE_ID,
    num_test=NUM_TEST,
    max_new_tokens=MAX_NEW_TOKENS,
    print_examples=PRINT_EXAMPLES,
    device="auto",
    save_results=SAVE_RESULTS,
)

resources = icl.load_experiment_resources(no_icl_args)
no_icl_df, no_icl_summary = icl.run_no_icl_experiment(resources, no_icl_args, save=SAVE_RESULTS)
no_icl_summary


In [ ]:
# Cell 2: ICL/oracle experiment over the full selected MATH test split.
# This reuses the model and dataset loaded in Cell 1 when run in order.
if "icl" not in globals():
    from pathlib import Path
    import importlib.util
    SCRIPT_PATH = Path("ICL-demo.py").resolve()
    spec = importlib.util.spec_from_file_location("icl_demo", SCRIPT_PATH)
    icl = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(icl)

icl_args = icl.make_experiment_args(
    mode="oracle",
    model=MODEL if "MODEL" in globals() else "Qwen/Qwen2.5-7B-Instruct",
    math_config=MATH_CONFIG if "MATH_CONFIG" in globals() else "algebra",
    test_scope=TEST_SCOPE if "TEST_SCOPE" in globals() else "full",
    split_index=SPLIT_INDEX if "SPLIT_INDEX" in globals() else 0,
    example_id=EXAMPLE_ID if "EXAMPLE_ID" in globals() else 0,
    num_test=NUM_TEST if "NUM_TEST" in globals() else None,
    num_contexts=20,
    k=4,
    max_new_tokens=MAX_NEW_TOKENS if "MAX_NEW_TOKENS" in globals() else 768,
    print_examples=PRINT_EXAMPLES if "PRINT_EXAMPLES" in globals() else 5,
    device="auto",
    save_results=SAVE_RESULTS if "SAVE_RESULTS" in globals() else True,
)

if "resources" not in globals():
    resources = icl.load_experiment_resources(icl_args)

icl_df, success_matrix_df, icl_summary, contexts = icl.run_oracle_context_experiment(
    resources,
    icl_args,
    save=icl_args.save_results,
)
icl_summary
